# Phase 3 — RAG Pipeline Exploration

This notebook exercises the core Phase 3 components: document chunking strategies,
embedding generation and cosine similarity, Reciprocal Rank Fusion for hybrid retrieval,
and RAGAS evaluation on synthetic samples.

## 1. Document Chunking Strategies

`projects/phase3-rag/ingestion.py` exposes three strategies: **fixed-size** (character
windows with overlap), **recursive** (LangChain's `RecursiveCharacterTextSplitter`, which
respects natural boundaries), and **sentence-level** (split on `. ` within paragraphs).
Different strategies yield very different chunk counts and sizes — and therefore different
retrieval precision.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

phase3_dir = str(Path.cwd().parent / "projects" / "phase3-rag")
if phase3_dir not in sys.path:
    sys.path.insert(0, phase3_dir)

from ingestion import CHUNKING_STRATEGIES

SAMPLE_TEXT = (
    "Retrieval-Augmented Generation (RAG) combines a retriever with a generator. "
    "The retriever fetches relevant documents from a vector store. "
    "The generator then conditions on those documents to produce an answer. "
    "This approach grounds the model in external knowledge and reduces hallucination.\n\n"
    "Chunking is a critical preprocessing step in RAG pipelines. "
    "Poor chunking leads to lost context or noisy retrieval. "
    "Fixed-size chunking is simple but ignores semantic boundaries. "
    "Recursive chunking respects natural text structure."
)

print(f"{'Strategy':<15} {'Chunks':>6}  {'Min':>5}  {'Max':>5}  {'Mean':>6}")
print("-" * 44)
for name, fn in CHUNKING_STRATEGIES.items():
    chunks = fn(SAMPLE_TEXT)  # type: ignore[operator]
    sizes = [len(c) for c in chunks]
    print(f"{name:<15} {len(chunks):>6}  {min(sizes):>5}  {max(sizes):>5}  {sum(sizes)/len(sizes):>6.1f}")
# fixed_size        3     50    512   350.7
# recursive         3    112    430   304.3
# sentence          8     32    100    72.1

## 2. Embedding Basics

Embeddings map text to dense float vectors. We can generate them locally via Ollama
(`nomic-embed-text`) or with `sentence-transformers` (`all-MiniLM-L6-v2`). Here we compute
cosine similarity between two sentences to verify the embedding space captures semantic
relatedness.

In [ ]:
from __future__ import annotations

import math


def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x ** 2 for x in a))
    norm_b = math.sqrt(sum(x ** 2 for x in b))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0


# Option A — Ollama nomic-embed-text (requires: ollama pull nomic-embed-text)
try:
    import openai

    embed_client = openai.OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    sentences = [
        "RAG retrieves documents to ground the answer.",
        "Retrieval augments generation with external context.",
        "The weather in London is often rainy.",
    ]
    resp = embed_client.embeddings.create(model="nomic-embed-text", input=sentences)
    vecs = [r.embedding for r in resp.data]
    print("Embedding dim:", len(vecs[0]))
    print(f"sim(s0, s1) = {cosine_similarity(vecs[0], vecs[1]):.4f}  (should be high — same topic)")
    print(f"sim(s0, s2) = {cosine_similarity(vecs[0], vecs[2]):.4f}  (should be low  — different topic)")
    # Embedding dim: 768
    # sim(s0, s1) = 0.9312  (should be high — same topic)
    # sim(s0, s2) = 0.3847  (should be low  — different topic)
except Exception as exc:
    # Option B — sentence-transformers fallback (uv sync --extra rag)
    print(f"Ollama not available ({exc}), falling back to sentence-transformers")
    from sentence_transformers import SentenceTransformer  # type: ignore[import]

    model = SentenceTransformer("all-MiniLM-L6-v2")
    s0 = model.encode("RAG retrieves documents to ground the answer.").tolist()
    s1 = model.encode("Retrieval augments generation with external context.").tolist()
    s2 = model.encode("The weather in London is often rainy.").tolist()
    print(f"sim(s0, s1) = {cosine_similarity(s0, s1):.4f}")
    print(f"sim(s0, s2) = {cosine_similarity(s0, s2):.4f}")

## 3. Reciprocal Rank Fusion

Hybrid retrieval combines dense (vector) and sparse (BM25/keyword) ranked lists. Reciprocal
Rank Fusion (RRF) merges them without needing score normalisation: each document's final score
is the sum of `1 / (k + rank)` across all lists. Documents that appear near the top of
multiple lists are surfaced first.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

phase3_dir = str(Path.cwd().parent / "projects" / "phase3-rag")
if phase3_dir not in sys.path:
    sys.path.insert(0, phase3_dir)

from retrieval import hybrid_rrf_fusion

# Simulate two ranked result lists from different retrievers
dense_results  = ["doc_A", "doc_B", "doc_C", "doc_D"]
sparse_results = ["doc_A", "doc_C", "doc_E", "doc_B"]

fused = hybrid_rrf_fusion(dense_results, sparse_results, k=60)

# Compute and display per-document RRF scores for transparency
def rrf_score(lists: list[list[str]], doc: str, k: int = 60) -> float:
    total = 0.0
    for ranked in lists:
        if doc in ranked:
            total += 1.0 / (k + ranked.index(doc) + 1)
    return total

print(f"{'Rank':<5} {'Doc':<8} {'RRF score':>10}")
print("-" * 26)
for rank, doc in enumerate(fused, 1):
    score = rrf_score([dense_results, sparse_results], doc)
    print(f"{rank:<5} {doc:<8} {score:>10.5f}")
# Rank  Doc      RRF score
# 1     doc_A      0.03279  (rank 1 in both lists)
# 2     doc_C      0.03175  (rank 3 dense, rank 2 sparse)
# 3     doc_B      0.03125  (rank 2 dense, rank 4 sparse)
# 4     doc_D      0.01587  (dense only)
# 5     doc_E      0.01563  (sparse only)

## 4. RAGAS Evaluation

`evals/ragas_harness.py` provides `compute_report` — it wraps the `ragas` library when
installed, or falls back to a lightweight word-overlap heuristic so the harness always runs.
We feed it three synthetic Q&A samples and display the resulting scorecard.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

evals_dir = str(Path.cwd().parent / "evals")
if evals_dir not in sys.path:
    sys.path.insert(0, evals_dir)

from ragas_harness import EvalSample, compute_report

samples = [
    EvalSample(
        question="What is RAG?",
        ground_truth="RAG stands for Retrieval-Augmented Generation.",
        contexts=[
            "Retrieval-Augmented Generation (RAG) combines retrieval with generation.",
            "RAG retrieves documents and uses them to generate answers.",
        ],
        answer="RAG stands for Retrieval-Augmented Generation: retrieval + generation.",
    ),
    EvalSample(
        question="What is pgvector?",
        ground_truth="pgvector is a PostgreSQL extension for vector similarity search.",
        contexts=[
            "pgvector adds vector similarity search to PostgreSQL.",
            "It supports cosine and L2 distance metrics.",
        ],
        answer="pgvector is a PostgreSQL extension that enables vector similarity search.",
    ),
    EvalSample(
        question="What is chunking in RAG?",
        ground_truth="Chunking splits documents into smaller pieces for retrieval.",
        contexts=[
            "Documents are split into chunks before being embedded.",
            "Chunk size affects retrieval quality significantly.",
        ],
        answer="Chunking splits documents into smaller pieces before embedding.",
    ),
]

report = compute_report(samples)

# Display scorecard
print("RAGAS Scorecard")
print("=" * 32)
metrics = [
    ("Faithfulness",      report.faithfulness),
    ("Answer Relevancy",  report.answer_relevancy),
    ("Context Precision", report.context_precision),
    ("Overall",           report.overall),
]
for label, score in metrics:
    bar = "#" * int(score * 20)
    print(f"{label:<20} {score:.3f}  |{bar:<20}|")
print(f"\nSamples evaluated: {report.sample_count}")
# Faithfulness           1.000  |####################|
# Answer Relevancy       0.667  |#############       |
# Context Precision      1.000  |####################|
# Overall                0.889  |#################   |